# D2C Robust Evaluation (Google Colab)

This notebook runs a 50-sample evaluation comparing **Vanilla**, **Speech Act (Original)**, and **Speech Act Surgical** using two different judge models (4B and 7B).

### 1. Setup Environment
Install Ollama and project dependencies. We force GPU support by checking nvidia drivers.

In [ ]:
# 1. Install discovery tools and zstd BEFORE Ollama
print("Installing system dependencies...")
!apt-get update && apt-get install -y zstd pciutils lshw

# 2. Install Ollama - it should now see the GPU
print("Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

import os
import subprocess
import time

# 3. Explicitly point to NVIDIA drivers
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:/usr/local/cuda/lib64'
os.environ['NVIDIA_VISIBLE_DEVICES'] = 'all'

print("Starting Ollama server...")
!nohup ollama serve > ollama.log 2>&1 &
time.sleep(15)

# 4. Final verification
print("NVIDIA-SMI Check:")
!nvidia-smi

print("\nOllama GPU Recognition Check (Look for 'pci' or 'cuda' in logs):")
!grep -E "pci|cuda|gpu" ollama.log || echo "No GPU mention in logs yet"

!pip install pandas requests sentence-transformers tqdm

### 2. Pull Models
We use 1.7B for generation and 4B/7B for judging.

In [ ]:
!ollama pull qwen3:1.7b
!ollama pull qwen3.5:4b
!ollama pull qwen2.5:7b

In [ ]:
import os
import shutil

# 1. Clean up old directory if it exists
repo_path = '/content/disagree-to-clarify'
if os.path.exists(repo_path):
    print(f"Overwriting existing directory: {repo_path}")
    shutil.rmtree(repo_path)

# 2. Clone fresh (Replace <TOKEN> with your PAT)
!git clone -b javier https://<TOKEN>@github.com/jashparekh1/disagree-to-clarify.git
%cd disagree-to-clarify

os.environ['PYTHONPATH'] = os.getcwd()
print("Current Directory:", os.getcwd())
!ls

### 4. Download Datasets
Download Clamber, Qulac, and ClariQ to the local disk.

In [ ]:
!python3 -m eval.datasets.download

### 5. Run Evaluation
This will take approximately 15-20 minutes for 50 samples.

In [ ]:
os.environ['PYTHONPATH'] = os.getcwd()
!python3 -m scripts.colab_detailed_eval

### 5. Download Results

In [ ]:
from google.colab import files
for f in ["colab_detailed_results.csv", "evaluation_audit.txt"]:
    if os.path.exists(f):
        files.download(f)